In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gym
import d4rl
import argparse
import pickle
import sys
import tqdm
import importlib
import os
from matplotlib import pyplot as plt

from common.normalizer import StandardNormalizer
# sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from common.normalizer import StandardNormalizer
from common import util
from models.transition_model import TransitionModel
from models.d4rl_world_model import D4RLWorldModel
from common.buffer import ReplayBuffer
from common.functional import dict_batch_generator
from scoring import crps_evaluation



No module named 'flow'
/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)
No module named 'carla'
pybullet build time: Jan 29 2025 23:19:57


RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xe . Check the section C-API incompatibility at the Troubleshooting ImportError section at https://numpy.org/devdocs/user/troubleshooting-importerror.html#c-api-incompatibility for indications on how to solve this problem .

numpy.core.multiarray failed to import


In [15]:
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
# data_path = "/abiomed/intermediate_data_d4rl/hopper-expert-v0_noisy_0.1_unnorm.pkl"
data_path = ""
env_name = 'hopper-expert-v0'
env = gym.make(env_name)
if data_path == "":
    dataset = d4rl.qlearning_dataset(env)
else:
    print('loading')
    with open(data_path, 'rb') as f:
        dataset = pickle.load(f)
    dataset = {key: torch.tensor(dataset[key], dtype = torch.float32) for key in dataset.keys()}

obs_shape = env.observation_space.shape
action_dim = np.prod(env.action_space.shape)   

offline_buffer = ReplayBuffer(
            buffer_size=len(dataset["observations"]),
            obs_shape=obs_shape,
            obs_dtype=np.float32,
            action_dim=action_dim,
            action_dtype=np.float32
        )

offline_buffer.load_dataset(dataset)

load datafile: 100%|██████████| 5/5 [00:04<00:00,  1.05it/s]


In [16]:
transition_params= {
        "model_batch_size": 256,
        "use_weight_decay": True,
        "optimizer_class": "Adam",
        "learning_rate": 0.001,
        "holdout_ratio": 0.2,
        "inc_var_loss": True,
        "model": {
            "hidden_dims": [200, 200, 200, 200],
            "decay_weights": [0.000025, 0.00005, 0.000075, 0.000075, 0.0001],
            "act_fn": "swish",
            "out_act_fn": "identity",
            "num_elite": 5,
            "ensemble_size": 7
        }
}

mopo_params = {
        "max_epoch": 125,
        "rollout_batch_size": 50000,
        "rollout_mini_batch_size": 10000,
        "model_retain_epochs": 1,
        "num_env_steps_per_epoch": 1000,
        "train_model_interval": 250,
        "max_trajectory_length": 1000,
        "eval_interval": 1000,
        "num_eval_trajectories": 10,
        "snapshot_interval": 2000,
        "model_env_ratio": 0.95,
        "max_model_update_epochs_to_improve": 5,
        "max_model_train_iterations": "None",
        'model_batch_size': 256,
        "rollout_batch_size":50000,
        "rollout_mini_batch_size":1000,
        "model_retain_epochs":1,
        "num_env_steps_per_epoch":1000,
        "max_epoch":100000,
        "max_model_update_epochs_to_improve":5,
        "max_model_train_iterations":np.inf,
        "hold_out_ratio":0.1,
    }
params = { **transition_params, **mopo_params }

In [17]:
task = env_name.split('-')[0]
import_path = f"static_fns.{task}"
static_fns = importlib.import_module(import_path).StaticFns
transition_model = TransitionModel(
        obs_space=env.observation_space,
        action_space=env.action_space,
        static_fns=static_fns,
        lr=transition_params['learning_rate'],
        device=device,
        **transition_params
    )

transition device cuda:3


In [18]:
def learn_dynamics(dynamics_model, offline_buffer, params=params):
        # get train and eval data
        model_tot_train_timesteps = 0
        max_sample_size = offline_buffer.get_size
        num_train_data = int(max_sample_size * (1.0 - params["holdout_ratio"]))
        env_data = offline_buffer.sample_all()
        train_data, eval_data = {}, {}
        for key in env_data.keys():
            train_data[key] = env_data[key][:num_train_data]
            eval_data[key] = env_data[key][num_train_data:]
            
        dynamics_model.reset_normalizers()
        dynamics_model.update_normalizer(train_data['observations'], train_data['actions'])

        # train model
        model_train_iters = 0
        model_train_epochs = 0
        num_epochs_since_prev_best = 0
        break_training = False
        dynamics_model.reset_best_snapshots()

        # init eval_mse_losses
        print("Start training dynamics")
        eval_mse_losses, _ = dynamics_model.eval_data(eval_data, update_elite_models=False)
        print("loss/model_eval_mse_loss", eval_mse_losses.mean(), model_tot_train_timesteps)
        updated = dynamics_model.update_best_snapshots(eval_mse_losses)
       
        while not break_training:
            # starttime = time.time()
            for train_data_batch in dict_batch_generator(train_data, params["model_batch_size"]):
                train_data_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in train_data_batch.items()}
                model_log_infos = dynamics_model.update(train_data_batch)
                model_train_iters += 1
                model_tot_train_timesteps += 1

            eval_mse_losses, _ = dynamics_model.eval_data(eval_data, update_elite_models=False)
            print("loss/model_eval_mse_loss", eval_mse_losses.mean(), model_tot_train_timesteps)
            # print('elapsed time',time.time() - starttime)
            updated = dynamics_model.update_best_snapshots(eval_mse_losses)
            num_epochs_since_prev_best += 1
            if updated:
                model_train_epochs += num_epochs_since_prev_best
                num_epochs_since_prev_best = 0
            if num_epochs_since_prev_best >= params["max_model_update_epochs_to_improve"] or model_train_iters > params["max_model_train_iterations"]\
                    or model_tot_train_timesteps > 800000:
                break
            
        
        #look at load_best_snapshots and model_best_snapshots
        dynamics_model.load_best_snapshots()

        # evaluate data to update the elite models
        dynamics_model.eval_data(eval_data, update_elite_models=True)
        model_log_infos['misc/norm_obs_mean'] = torch.mean(torch.Tensor(dynamics_model.obs_normalizer.mean)).item()
        model_log_infos['misc/norm_obs_var'] = torch.mean(torch.Tensor(dynamics_model.obs_normalizer.var)).item()
        model_log_infos['misc/norm_act_mean'] = torch.mean(torch.Tensor(dynamics_model.act_normalizer.mean)).item()
        model_log_infos['misc/norm_act_var'] = torch.mean(torch.Tensor(dynamics_model.act_normalizer.var)).item()
        model_log_infos['misc/model_train_epochs'] = model_train_epochs
        model_log_infos['misc/model_train_train_steps'] = model_train_iters
        return dynamics_model

In [19]:
trained_dynamics_model = learn_dynamics(transition_model, offline_buffer, params)
# env_name2 = env_name + "_noisy"
torch.save(trained_dynamics_model.state_dict(), f'saved_models/{env_name}/dynamics_model.pt')

Start training dynamics


RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float